# 🧘 Yoga Pose Similarity Search (Text + Image Search)
### Multimodal embeddings with CLIP + Milvus Vector Database

This version upgrades your ResNet-only pipeline to **CLIP**, so you can:
- Search with an **image** (image → similar images)
- Search with **text** (text → relevant images)
- Optionally search with **image + text** (weighted fusion)

All data + the Milvus DB + checkpoints are stored on Google Drive so you can resume after disconnects.

---
## Cell 1 — Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

# ⚙️ Change this if you want a different folder
DRIVE_ROOT = '/content/drive/MyDrive/YogaPoseSearch'

DRIVE_DATA       = os.path.join(DRIVE_ROOT, 'dataset')
DRIVE_DB         = os.path.join(DRIVE_ROOT, 'yoga_milvus.db')
DRIVE_CHECKPOINT = os.path.join(DRIVE_ROOT, 'checkpoint.txt')
DRIVE_KAGGLE     = os.path.join(DRIVE_ROOT, 'kaggle.json')

for d in [DRIVE_ROOT, DRIVE_DATA]:
    os.makedirs(d, exist_ok=True)

print('Google Drive mounted ✓')
print('Drive root :', DRIVE_ROOT)
print('Dataset    :', DRIVE_DATA)
print('Milvus DB  :', DRIVE_DB)
print('Checkpoint :', DRIVE_CHECKPOINT)

---
## Cell 2 — Install Dependencies

We use **OpenAI CLIP** (via `open_clip_torch`) for multimodal embeddings.

In [ ]:
!pip install -q milvus-lite pymilvus open_clip_torch torch torchvision pillow kaggle flask pyngrok

---
## Cell 3 — Kaggle Auth & Dataset Download (Drive-backed)

In [ ]:
import shutil, glob
from google.colab import files as colab_files

KAGGLE_DIR = '/root/.kaggle'
os.makedirs(KAGGLE_DIR, exist_ok=True)

if os.path.exists(DRIVE_KAGGLE):
    shutil.copy(DRIVE_KAGGLE, f'{KAGGLE_DIR}/kaggle.json')
    print('kaggle.json loaded from Drive ✓')
else:
    print('kaggle.json not found on Drive — please upload it now:')
    uploaded = colab_files.upload()
    with open(f'{KAGGLE_DIR}/kaggle.json', 'wb') as fh:
        fh.write(uploaded['kaggle.json'])
    shutil.copy(f'{KAGGLE_DIR}/kaggle.json', DRIVE_KAGGLE)
    print('kaggle.json saved to Drive ✓')

os.chmod(f'{KAGGLE_DIR}/kaggle.json', 0o600)

existing = glob.glob(os.path.join(DRIVE_DATA, '**/train'), recursive=True)
if existing:
    DATASET_DIR = existing[0]
    print('Dataset already on Drive — skipping download ✓')
else:
    print('Downloading dataset to Drive...')
    !kaggle datasets download -d shrutisaxena/yoga-pose-image-classification-dataset -p "{DRIVE_DATA}" --unzip
    existing = glob.glob(os.path.join(DRIVE_DATA, '**/train'), recursive=True)
    DATASET_DIR = existing[0] if existing else DRIVE_DATA

print('Dataset root:', DATASET_DIR)
print('Classes found:', len(os.listdir(DATASET_DIR)))

---
## Cell 4 — ⚙️ Configuration

In [ ]:
DB_PATH    = DRIVE_DB
COLLECTION = 'yoga_clip_embeddings'

# CLIP embedding dimension depends on model. For ViT-B-32, it's 512.
EMB_DIM = 512

BATCH_SIZE = 128
TOP_N = 5

print('Config set ✓')
print('DB_PATH   :', DB_PATH)
print('COLLECTION:', COLLECTION)
print('EMB_DIM   :', EMB_DIM)

---
## Cell 5 — Load CLIP (Image+Text) Embedding Model

In [ ]:
import torch
import open_clip
from PIL import Image
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

# Model choice: ViT-B-32 is a good balance for Colab
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
clip_model = clip_model.to(device).eval()
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')

print('CLIP loaded ✓')
# Sanity check embedding dim
with torch.no_grad():
    t = clip_tokenizer(['test']).to(device)
    e = clip_model.encode_text(t)
print('CLIP text embedding dim:', e.shape[-1])

---
## Cell 6 — Embedding Functions (Image + Text)

In [ ]:
def l2_normalize(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x) + 1e-10)

def get_image_embedding(image_path: str) -> np.ndarray:
    img = Image.open(image_path).convert('RGB')
    img_t = clip_preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = clip_model.encode_image(img_t).squeeze().cpu().numpy()
    emb = l2_normalize(emb).astype(np.float32)
    return emb

def get_text_embedding(text: str) -> np.ndarray:
    tokens = clip_tokenizer([text]).to(device)
    with torch.no_grad():
        emb = clip_model.encode_text(tokens).squeeze().cpu().numpy()
    emb = l2_normalize(emb).astype(np.float32)
    return emb

print('get_image_embedding() + get_text_embedding() defined ✓')

---
## Cell 7 — Build / Resume Milvus DB (Store Image Vectors)

We store **image embeddings** only, but you can query Milvus with either:
- an image embedding (image → images)
- a text embedding (text → images)
- a fused embedding (image+text → images)

Checkpointing is stored on Drive so you can resume after disconnects.

In [ ]:
from pymilvus import MilvusClient
import time

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.jpe'}

# Load checkpoint
if os.path.exists(DRIVE_CHECKPOINT):
    with open(DRIVE_CHECKPOINT, 'r') as fh:
        already_done = set(line.strip() for line in fh if line.strip())
    print('Checkpoint loaded ✓', len(already_done))
else:
    already_done = set()
    print('No checkpoint found — starting fresh.')

client = MilvusClient(DB_PATH)

# If collection exists but has wrong dimension, recreate
if client.has_collection(COLLECTION):
    info = client.describe_collection(COLLECTION)
    dim_ok = (info.get('dimension') == EMB_DIM)
    if not dim_ok:
        client.drop_collection(COLLECTION)
        print('Dropped collection due to dimension mismatch')

if not client.has_collection(COLLECTION):
    client.create_collection(collection_name=COLLECTION, dimension=EMB_DIM, metric_type='COSINE', auto_id=True)
    print('Created collection', COLLECTION)

# Collect images
all_images = [
    os.path.join(root, f)
    for root, _, files in os.walk(DATASET_DIR)
    for f in files
    if os.path.splitext(f)[1].lower() in VALID_EXTS
]
pending = [p for p in all_images if p not in already_done]
print('Total images:', len(all_images), '| Pending:', len(pending))

data_buffer, done_buffer = [], []
inserted, skipped = 0, 0
start = time.time()
checkpoint_fh = open(DRIVE_CHECKPOINT, 'a')

try:
    for i, fpath in enumerate(pending):
        label = os.path.basename(os.path.dirname(fpath))
        try:
            emb = get_image_embedding(fpath)
        except Exception:
            skipped += 1
            continue

        data_buffer.append({'vector': emb, 'label': label, 'image_path': fpath})
        done_buffer.append(fpath)

        if len(data_buffer) >= BATCH_SIZE:
            client.insert(COLLECTION, data_buffer)
            checkpoint_fh.write('
'.join(done_buffer) + '
')
            checkpoint_fh.flush()
            inserted += len(data_buffer)
            data_buffer, done_buffer = [], []

        if (i + 1) % 500 == 0:
            print(f'[{i+1}/{len(pending)}] inserted={inserted} skipped={skipped} elapsed={time.time()-start:.0f}s')

    if data_buffer:
        client.insert(COLLECTION, data_buffer)
        checkpoint_fh.write('
'.join(done_buffer) + '
')
        checkpoint_fh.flush()
        inserted += len(data_buffer)
finally:
    checkpoint_fh.close()
    client.close()

print('✅ Build complete. Inserted:', inserted, 'Skipped:', skipped, 'Time:', round(time.time()-start,1),'s')

---
## Cell 8 — Search APIs: image query, text query, and fused query

In [ ]:
def _milvus_search(query_vec: np.ndarray, top_n: int = TOP_N):
    c = MilvusClient(DB_PATH)
    res = c.search(
        collection_name=COLLECTION,
        data=[query_vec.astype(np.float32)],
        limit=top_n,
        output_fields=['label', 'image_path'],
        search_params={'metric_type': 'COSINE'},
    )
    c.close()
    return [
        {'score': round(float(h['distance']), 4),
         'label': h['entity']['label'],
         'image_path': h['entity']['image_path']}
        for h in res[0]
    ]

def search_by_image(image_path: str, top_n: int = TOP_N):
    return _milvus_search(get_image_embedding(image_path), top_n=top_n)

def search_by_text(text: str, top_n: int = TOP_N):
    return _milvus_search(get_text_embedding(text), top_n=top_n)

def search_by_image_and_text(image_path: str, text: str, alpha: float = 0.5, top_n: int = TOP_N):
    # alpha=1.0 → only image; alpha=0.0 → only text
    img_v = get_image_embedding(image_path)
    txt_v = get_text_embedding(text)
    q = alpha * img_v + (1.0 - alpha) * txt_v
    q = l2_normalize(q).astype(np.float32)
    return _milvus_search(q, top_n=top_n)

print('Search functions ready ✓')

---
## Cell 9 — Quick demo: Text → Images

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

query = 'a person doing downward dog yoga pose'
hits = search_by_text(query, top_n=5)
print('Query:', query)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, hit in zip(axes, hits):
    ax.imshow(mpimg.imread(hit['image_path']))
    ax.set_title(f"{hit['label']}\nscore={hit['score']}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()